# Model Training Example

This notebook shows how to instantiate the configured model, train and evaluate it, export ONNX, and run inference. Expensive steps are guarded by flags so opening the notebook does not immediately download the backbone or train the model.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

from intent_classifier.dataset import RequestDataset, create_data_loaders, split_ids
from intent_classifier.inference import IntentEstimator, postprocess_predictions, predict_onnx
from intent_classifier.model import create_model, export_onnx, load_onnx
from intent_classifier.preprocessing import export_tokenizer, load_tokenizer
from intent_classifier.settings import load_model_config, load_train_config
from intent_classifier.train import (
    evaluate,
    fit,
    quantize_onnx,
    save_calibration,
    save_thresholds,
    tune_thresholds,
)
from intent_classifier.utils import save_json, set_seed

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_CONFIG_PATH = PROJECT_ROOT / "intent_classifier/config/model_config.yaml"
TRAIN_CONFIG_PATH = PROJECT_ROOT / "intent_classifier/config/train_config.yaml"

model_config = load_model_config(MODEL_CONFIG_PATH)
train_config = load_train_config(TRAIN_CONFIG_PATH)
artifact_dir = PROJECT_ROOT / train_config.artifact_dir
dataset_path = PROJECT_ROOT / train_config.dataset_csv

set_seed(train_config.seed)
model_config.head_names, artifact_dir, dataset_path

## Load Dataset and Tokenizer

Set `EXPORT_TOKENIZER = True` the first time you want to download and save the Hugging Face tokenizer locally. Production inference should load the tokenizer from artifacts with `local_files_only=True`.

In [ ]:
EXPORT_TOKENIZER = False

if EXPORT_TOKENIZER:
    tokenizer_dir = export_tokenizer(model_config.backbone, PROJECT_ROOT / model_config.backbone.local_tokenizer_path)
else:
    tokenizer_dir = PROJECT_ROOT / model_config.backbone.local_tokenizer_path

print("Tokenizer directory:", tokenizer_dir)
print("Dataset rows:", len(RequestDataset.from_csv(dataset_path, model_config)))

In [ ]:
# This cell requires tokenizer files to exist. Run the previous cell with EXPORT_TOKENIZER=True first.
LOAD_TOKENIZER = False

dataset = RequestDataset.from_csv(dataset_path, model_config)
splits = split_ids(dataset.frame, train_config.splits, seed=train_config.seed)

if LOAD_TOKENIZER:
    tokenizer = load_tokenizer(tokenizer_dir, local_files_only=True)
    loaders = create_data_loaders(
        dataset,
        tokenizer,
        model_config,
        splits,
        batch_size=train_config.training.phase_1.batch_size,
    )
    batch = next(iter(loaders["train"]))
    print(batch["input_ids"].shape, batch["attention_mask"].shape)
else:
    tokenizer = None
    loaders = None
    print("Tokenizer loading skipped.")

## Instantiate and Train

The following cells use the real configured backbone. Set `RUN_TRAINING = True` when you are ready to download the model and train. With only 100 synthetic samples, this is a smoke test, not a quality model.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    model = fit(train_config, model_config)
else:
    model = None
    print("Training skipped. Set RUN_TRAINING=True to run fit(...).")

In [ ]:
if model is not None and loaders is not None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    metrics = evaluate(model.to(device), loaders["test"], model_config, train_config.loss, device)
    metrics
else:
    metrics = None
    print("Evaluation skipped because model or loaders are not available.")

## Export ONNX and Quantize

After training, export raw logits to ONNX and optionally quantize to INT8.

In [ ]:
RUN_ONNX_EXPORT = False

onnx_path = artifact_dir / "model.onnx"
int8_path = artifact_dir / "model.int8.onnx"

if RUN_ONNX_EXPORT and model is not None:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    export_onnx(model, model_config, onnx_path)
    quantize_onnx(onnx_path, int8_path)
    print("Exported:", onnx_path, int8_path)
else:
    print("ONNX export skipped.")

## Thresholds and Calibration Artifacts

This small example writes neutral calibration values and default thresholds. In production, fit calibration and tune thresholds on the validation split.

In [ ]:
WRITE_DEFAULT_RUNTIME_ARTIFACTS = False

if WRITE_DEFAULT_RUNTIME_ARTIFACTS:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    calibration = {
        "version": "calibration_v1",
        "fitted_on": "example_defaults",
        "heads": {
            head.name: {
                "method": "per_label_temperature_scaling",
                "temperatures": {label: 1.0 for label in head.labels},
            }
            for head in model_config.heads
        },
    }
    thresholds = {
        "version": "thresholds_v1",
        "heads": {
            head.name: {label: {"activate": 0.5} for label in head.labels}
            for head in model_config.heads
        },
    }
    save_calibration(calibration, artifact_dir / "calibration.json")
    save_thresholds(thresholds, artifact_dir / "thresholds.json")
    save_json({"heads": list(model_config.head_names)}, artifact_dir / "evaluation_report.json")
    print("Wrote default runtime artifacts.")
else:
    print("Default runtime artifact write skipped.")

## Run ONNX Inference

This path expects a trained/exported ONNX model and tokenizer artifacts under `artifact_dir`.

In [ ]:
RUN_INFERENCE = False
example_text = "hazme un presupuesto y agenda una visita para manana"

if RUN_INFERENCE:
    estimator = IntentEstimator(artifact_dir)
    prediction = estimator.predict(example_text)
    prediction
else:
    print("Inference skipped. Set RUN_INFERENCE=True after exporting artifacts.")

## Direct Postprocessing Demo

The cell below does not require ONNX artifacts. It demonstrates how logits become probabilities and active labels.

In [ ]:
fake_logits = {
    "business": np.array([[3.0, -2.0, 2.5, -3.0, -3.0, -3.0, -3.0, -3.0, -1.0, -1.0]]),
    "undesired": np.array([[-3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]]),
}
default_thresholds = {
    "heads": {
        head.name: {label: {"activate": 0.5} for label in head.labels}
        for head in model_config.heads
    }
}

postprocess_predictions(fake_logits, model_config, calibration={"heads": {}}, thresholds=default_thresholds)